# 01 — Kapan Fine-tuning, dan LoRA dari Nol

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/10-llm-finetuning/01_kapan_finetuning_dan_lora_dari_nol.ipynb)

Di modul 08–09 kita mengarahkan LLM **tanpa mengubah bobotnya**: lewat prompt dan konteks (RAG). **Fine-tuning** mengubah bobot model dengan melatihnya lagi pada data kita. Notebook ini menjawab dua pertanyaan sebelum kita fine-tune LLM sungguhan di notebook 02:

1. **Kapan** fine-tuning layak dipakai, dibandingkan prompting atau RAG?
2. Bagaimana fine-tuning bisa dilakukan **murah**? Jawabannya adalah **LoRA** (*Low-Rank Adaptation*), yang kita bangun dari nol dengan PyTorch lalu cocokkan dengan library `peft`.

Isi:
1. Prompting vs RAG vs fine-tuning.
2. Hitung-hitungan memori: mengapa full fine-tuning LLM mahal.
3. Intuisi low-rank dengan SVD.
4. `LoRALinear` dari nol, diverifikasi terhadap `peft`.
5. Eksperimen: adaptasi ke tugas baru (full fine-tuning vs LoRA vs hanya layer terakhir) dan *catastrophic forgetting*.
6. Kuantisasi bobot (dasar QLoRA).

> Semua eksperimen memakai MLP kecil dan FashionMNIST, jadi cukup dengan CPU. Konsepnya persis sama dengan yang dipakai pada LLM miliaran parameter.

## 1. Prompting vs RAG vs Fine-tuning

| | Prompting (+ few-shot) | RAG | Fine-tuning |
|---|---|---|---|
| **Cocok untuk** | Tugas umum, prototipe | **Pengetahuan**: dokumen privat, sering berubah, perlu sitasi | **Perilaku**: format output tetap, gaya bahasa, tugas sempit bervolume tinggi |
| **Data yang dibutuhkan** | Beberapa contoh | Dokumen | Ratusan–ribuan contoh input → output berkualitas |
| **Biaya awal** | Hampir nol | Sedang (pipeline, index) | Tinggi (data, GPU, eksperimen) |
| **Biaya per permintaan** | Bisa tinggi (prompt panjang, model besar) | Sedang (konteks) | Bisa rendah (model kecil hasil fine-tune, prompt pendek) |
| **Update pengetahuan** | Ubah prompt | Ubah dokumen, langsung berlaku | Latih ulang |

Aturan praktis:
- **Mulai dari prompting.** Banyak masalah selesai di sini, dan kamu mendapat baseline serta set evaluasi.
- Masalahnya **model tidak tahu fakta tertentu**? Gunakan **RAG**, bukan fine-tuning. Fine-tuning tidak andal untuk "memasukkan" fakta baru dan sulit diperbarui.
- Masalahnya **perilaku** (format tidak konsisten, gaya salah, tugas spesifik yang ingin dijalankan model kecil yang murah dan cepat)? Pertimbangkan **fine-tuning**, dan ukur terhadap baseline prompting.
- Ketiganya bisa digabung: model hasil fine-tune tetap bisa diberi prompt yang baik dan konteks RAG.

## 2. Mengapa Full Fine-tuning LLM Mahal?

Saat melatih dengan optimizer Adam dan *mixed precision*, setiap parameter membutuhkan kira-kira **16 byte** memori GPU: bobot bf16 (2), gradien bf16 (2), salinan bobot fp32 (4), dan dua *state* Adam fp32 (4 + 4). Itu belum termasuk **aktivasi** (hasil antara forward pass yang disimpan untuk backprop), yang bergantung pada panjang sequence dan batch size.

Dengan **LoRA**, bobot asli **dibekukan** (hanya perlu disimpan, 2 byte/parameter di bf16), dan yang dilatih hanya adapter kecil (biasanya < 1% parameter). **QLoRA** melangkah lebih jauh: bobot beku disimpan dalam **4-bit** (±0,5 byte/parameter).

In [ ]:
import pandas as pd

BYTE_PER_PARAM = {
    "Full FT (Adam, mixed precision)": 16,
    "LoRA (bobot beku bf16)": 2,
    "QLoRA (bobot beku 4-bit)": 0.5,
}
PORSI_ADAPTER = 0.01      # anggap adapter LoRA = 1% parameter, dilatih penuh (16 byte/parameter)


def memori_gb(n_param, metode):
    total = n_param * BYTE_PER_PARAM[metode]
    if metode != "Full FT (Adam, mixed precision)":
        total += n_param * PORSI_ADAPTER * 16
    return total / 1e9


tabel = pd.DataFrame({metode: {f"{n / 1e9:g}B parameter": round(memori_gb(n, metode), 1)
                               for n in [0.5e9, 7e9, 70e9]} for metode in BYTE_PER_PARAM})
print("Perkiraan memori GPU (GB), belum termasuk aktivasi. Pembanding: T4 = 16 GB, A100 = 40/80 GB")
tabel

Full fine-tuning model 7B butuh ±112 GB (beberapa GPU A100), sedangkan QLoRA 7B muat di satu GPU T4 16 GB gratis di Colab. Inilah alasan LoRA/QLoRA menjadi cara standar fine-tuning LLM bagi individu dan tim kecil.

## 3. Intuisi Low-Rank

Fine-tuning mengubah matriks bobot `W` (ukuran `d × k`) menjadi `W + ΔW`. Hipotesis di balik LoRA: perubahan `ΔW` untuk menyesuaikan model ke tugas baru punya **rank rendah**, artinya bisa didekati dengan perkalian dua matriks kurus:

```
ΔW ≈ B · A      B: d × r,  A: r × k,  dengan r ≪ min(d, k)
```

Jumlah parameter turun dari `d·k` menjadi `r·(d + k)`. Untuk `d = k = 4096` dan `r = 8`: dari 16,8 juta menjadi 65 ribu (0,4%).

Kita lihat dengan **SVD** (*singular value decomposition*): matriks yang "sebenarnya" low-rank punya sedikit nilai singular besar, sisanya kecil. Memotong di rank `r` memberi pendekatan terbaik dengan `r` komponen.

In [ ]:
import matplotlib.pyplot as plt
import torch

torch.manual_seed(0)
d, k, rank_asli = 512, 512, 8
# ΔW sintetis: struktur rank 8 + sedikit noise
delta_w = torch.randn(d, rank_asli) @ torch.randn(rank_asli, k) / rank_asli ** 0.5 + 0.05 * torch.randn(d, k)

U, S, Vh = torch.linalg.svd(delta_w)
baris = []
for r in [1, 2, 4, 8, 16, 64]:
    pendekatan = (U[:, :r] * S[:r]) @ Vh[:r]
    galat = (torch.linalg.norm(delta_w - pendekatan) / torch.linalg.norm(delta_w)).item()
    baris.append({"rank r": r, "galat relatif": round(galat, 3), "parameter r·(d+k)": r * (d + k),
                  "% dari d·k": round(100 * r * (d + k) / (d * k), 2)})

plt.figure(figsize=(6, 3))
plt.semilogy(S[:40].numpy(), "o-")
plt.axvline(rank_asli - 0.5, color="gray", ls="--")
plt.title("Nilai singular ΔW (skala log)")
plt.xlabel("indeks")
plt.show()
pd.DataFrame(baris)

Setelah rank 8, galat hampir tidak turun lagi: sisa informasinya hanya noise. Dengan 3% parameter, kita menangkap hampir seluruh perubahan. LoRA tidak menghitung SVD; matriks `A` dan `B` langsung **dilatih** dengan gradient descent, dan rank `r` adalah hyperparameter.

## 4. `LoRALinear` dari Nol

Sebuah layer `nn.Linear` dibungkus menjadi:

```
h = W·x + (α / r) · B·A·x        W beku; hanya A dan B yang dilatih
```

Detail penting:
- **Inisialisasi:** `A` acak, `B` **nol**. Jadi di awal `B·A = 0` dan model berperilaku **persis** seperti model asli. Training dimulai dari titik yang sama dengan model pretrained.
- **Skala `α / r`:** `lora_alpha` (α) mengatur besarnya kontribusi adapter. Dengan skala ini, mengganti `r` tidak banyak mengubah learning rate yang cocok.
- **Merge:** setelah training, `W' = W + (α/r)·B·A` bisa dihitung sekali. Model kembali menjadi `nn.Linear` biasa, **tanpa tambahan latensi** saat inference.

In [ ]:
import math

import torch.nn as nn


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=4, alpha=8, dropout=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                     # bobot asli dibekukan
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # sama dengan inisialisasi di peft
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.dropout(x) @ self.A.T @ self.B.T) * self.scale

    @torch.no_grad()
    def merge(self):
        """Gabungkan adapter ke bobot: kembalikan nn.Linear biasa."""
        gabung = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None)
        gabung.weight.copy_(self.base.weight + self.scale * self.B @ self.A)
        if self.base.bias is not None:
            gabung.bias.copy_(self.base.bias)
        return gabung


# Uji: di awal, output LoRA harus identik dengan layer asli (karena B = 0)
lin = nn.Linear(16, 8)
lora = LoRALinear(lin, r=2)
x = torch.randn(3, 16)
print("Identik di awal:", torch.allclose(lin(x), lora(x)))
print("Parameter dilatih:", sum(p.numel() for p in lora.parameters() if p.requires_grad),
      "dari", sum(p.numel() for p in lora.parameters()))

## 5. Eksperimen: Adaptasi ke Tugas Baru

Skenario mini yang meniru fine-tuning LLM:
- **Pretraining (tugas A):** MLP dilatih pada 60.000 gambar FashionMNIST (pakaian & sepatu, 10 kelas). Ini "model dasar" kita.
- **Tugas baru (tugas B):** mengenali **angka tulisan tangan** (MNIST, 10 kelas), hanya dengan **5.000 contoh**. Format inputnya sama (gambar 28×28), tetapi domainnya berbeda, mirip LLM umum yang disesuaikan untuk tugas khusus dengan data terbatas.

Kita bandingkan beberapa cara adaptasi ke tugas B: hanya melatih layer terakhir, LoRA dengan rank berbeda, dan full fine-tuning. Yang diukur: akurasi tugas B, jumlah parameter yang dilatih, dan akurasi tugas A **setelah** adaptasi (apakah model "lupa"?).

In [ ]:
import copy
import os

import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

tf = transforms.ToTensor()
fashion_train = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
fashion_test = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
mnist_train = datasets.MNIST("data", train=True, download=True, transform=tf)
mnist_test = datasets.MNIST("data", train=False, download=True, transform=tf)
N_PRETRAIN, N_ADAPT, N_TEST = (3000, 1000, 1000) if QUICK_RUN else (60000, 5000, 10000)
EPOCH_PRETRAIN = 1 if QUICK_RUN else 3

data_pretrain = Subset(fashion_train, range(N_PRETRAIN))
data_adapt = Subset(mnist_train, torch.randperm(len(mnist_train))[:N_ADAPT].tolist())   # tugas B: data sedikit
test_a = Subset(fashion_test, range(N_TEST))
test_b = Subset(mnist_test, range(N_TEST))
print(f"pretrain (FashionMNIST): {len(data_pretrain)} | adaptasi (MNIST): {len(data_adapt)} | device: {device}")

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2, self.fc3 = nn.Linear(784, 256), nn.Linear(256, 128), nn.Linear(128, 10)

    def forward(self, x):
        x = x.flatten(1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))


def latih(model, data, epoch=1, lr=1e-3):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(params, lr=lr)
    model.train()
    for _ in range(epoch):
        for x, y in DataLoader(data, batch_size=128, shuffle=True):
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    return model


@torch.no_grad()
def akurasi(model, data):
    model.eval()
    benar = 0
    for x, y in DataLoader(data, batch_size=512):
        x, y = x.to(device), y.to(device)
        benar += (model(x).argmax(1) == y).sum().item()
    return benar / len(data)


pretrained = latih(MLP().to(device), data_pretrain, epoch=EPOCH_PRETRAIN)
print(f"Model pretrained | akurasi tugas A (FashionMNIST): {akurasi(pretrained, test_a):.3f} | "
      f"tugas B (MNIST, sebelum adaptasi): {akurasi(pretrained, test_b):.3f}")

Tanpa adaptasi, akurasi MNIST setara tebakan acak (±10%). Sekarang adaptasi dengan beberapa cara. Fungsi `pasang_lora` mengganti `fc1`, `fc2`, `fc3` dengan `LoRALinear`. Learning rate LoRA biasanya lebih besar daripada full fine-tuning karena parameter yang dilatih sedikit dan dimulai dari nol.

In [ ]:
def pasang_lora(model, r, alpha=None):
    for nama in ["fc1", "fc2", "fc3"]:
        setattr(model, nama, LoRALinear(getattr(model, nama), r=r, alpha=alpha or 2 * r))
    return model


def hanya_layer_terakhir(model):
    for nama, p in model.named_parameters():
        p.requires_grad = nama.startswith("fc3")
    return model


EPOCH_ADAPT = 1 if QUICK_RUN else 5
cara = {
    "hanya layer terakhir": (hanya_layer_terakhir, 1e-3),
    "LoRA r=1": (lambda m: pasang_lora(m, r=1), 5e-3),
    "LoRA r=4": (lambda m: pasang_lora(m, r=4), 5e-3),
    "LoRA r=16": (lambda m: pasang_lora(m, r=16), 5e-3),
    "full fine-tuning": (lambda m: m, 1e-3),
}

hasil, model_adaptasi = [], {}
for nama, (siapkan, lr) in cara.items():
    torch.manual_seed(0)
    m = siapkan(copy.deepcopy(pretrained))
    n_latih = sum(p.numel() for p in m.parameters() if p.requires_grad)
    latih(m, data_adapt, epoch=EPOCH_ADAPT, lr=lr)
    model_adaptasi[nama] = m
    hasil.append({"cara": nama, "parameter dilatih": n_latih,
                  "% parameter": round(100 * n_latih / sum(p.numel() for p in pretrained.parameters()), 2),
                  "akurasi tugas B": round(akurasi(m, test_b), 3),
                  "akurasi tugas A (setelah)": round(akurasi(m, test_a), 3)})
df_hasil = pd.DataFrame(hasil)
df_hasil

Cara membaca tabel:
- **Hanya layer terakhir** murah, tetapi kurang mampu: fitur di layer awal (hasil belajar pakaian) tidak bisa menyesuaikan diri dengan bentuk angka.
- **LoRA** mengubah *semua* layer lewat adapter kecil. Rank lebih besar = kapasitas lebih besar; dengan ±3–10% parameter, akurasinya biasanya mendekati full fine-tuning.
- Kolom **tugas A (setelah)** menunjukkan *catastrophic forgetting*: model yang diadaptasi ke tugas B bisa kehilangan kemampuan lamanya.

Keunggulan praktis LoRA: bobot asli **tidak berubah**. Lepas adapter, dan model kembali persis seperti semula. Satu model dasar bisa dipasangi banyak adapter untuk banyak tugas, masing-masing hanya beberapa MB.

In [ ]:
m_lora = model_adaptasi["LoRA r=4"]
tanpa_adapter = copy.deepcopy(m_lora)
for nama in ["fc1", "fc2", "fc3"]:
    setattr(tanpa_adapter, nama, getattr(tanpa_adapter, nama).base)       # lepas adapter

print(f"LoRA r=4 dengan adapter  | tugas A: {akurasi(m_lora, test_a):.3f} | tugas B: {akurasi(m_lora, test_b):.3f}")
print(f"LoRA r=4 adapter dilepas | tugas A: {akurasi(tanpa_adapter, test_a):.3f} (= model pretrained: "
      f"{akurasi(pretrained, test_a):.3f})")

# Merge: gabungkan adapter ke bobot, hasil harus identik
gabung = copy.deepcopy(m_lora)
for nama in ["fc1", "fc2", "fc3"]:
    setattr(gabung, nama, getattr(gabung, nama).merge())
x = next(iter(DataLoader(test_b, batch_size=64)))[0].to(device)
print("Output setelah merge identik:", torch.allclose(m_lora.eval()(x), gabung.eval()(x), atol=1e-5))

### Mencocokkan dengan Library `peft`

Dalam praktik kita memakai library [`peft`](https://github.com/huggingface/peft) dari Hugging Face (dipakai di notebook 02). `LoraConfig` berisi hyperparameter yang sama: `r`, `lora_alpha`, `lora_dropout`, dan `target_modules` (layer mana yang diberi adapter). Jumlah parameter yang dilatih harus sama dengan implementasi kita.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_model = get_peft_model(copy.deepcopy(pretrained),
                            LoraConfig(r=4, lora_alpha=8, target_modules=["fc1", "fc2", "fc3"]))
peft_model.print_trainable_parameters()

n_peft = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
n_kita = df_hasil.set_index("cara").loc["LoRA r=4", "parameter dilatih"]
print(f"peft: {n_peft:,} | implementasi kita: {n_kita:,} | sama: {n_peft == n_kita}")
print(peft_model.base_model.model.fc1)

## 6. Kuantisasi: Dasar QLoRA

**Kuantisasi** menyimpan bobot dengan bit lebih sedikit. Cara paling sederhana adalah *absmax* per blok: bagi bobot menjadi blok kecil (misalnya 64 angka), skalakan tiap blok agar nilai absolut terbesarnya pas di rentang integer, lalu bulatkan. Saat dipakai, integer dikalikan kembali dengan skalanya (*dequantize*).

QLoRA memakai format 4-bit **NF4** (*NormalFloat*), yang titik-titik nilainya disesuaikan dengan distribusi bobot jaringan saraf yang mirip distribusi normal. Idenya sama dengan contoh sederhana di bawah. Bobot dasar disimpan 4-bit dan dibekukan, sedangkan adapter LoRA dilatih dalam presisi lebih tinggi.

In [ ]:
def kuantisasi_absmax(w, bit, blok=64):
    """Kuantisasi lalu dequantize (untuk mengukur galat). Jumlah elemen harus kelipatan `blok`."""
    q_maks = 2 ** (bit - 1) - 1
    w_blok = w.reshape(-1, blok)
    skala = w_blok.abs().amax(dim=1, keepdim=True) / q_maks
    q = torch.round(w_blok / skala).clamp(-q_maks, q_maks)     # integer yang disimpan
    return (q * skala).reshape(w.shape)


baris = []
for bit in [16, 8, 4, 3, 2]:
    m = copy.deepcopy(pretrained)
    with torch.no_grad():
        for lin in [m.fc1, m.fc2, m.fc3]:
            if bit < 16:
                lin.weight.copy_(kuantisasi_absmax(lin.weight, bit))
    galat = (torch.linalg.norm(m.fc1.weight - pretrained.fc1.weight) / torch.linalg.norm(pretrained.fc1.weight)).item()
    baris.append({"bit": bit, "ukuran bobot vs fp32": f"{bit / 32:.0%}", "galat relatif fc1": round(galat, 4),
                  "akurasi tugas A": round(akurasi(m, test_a), 3)})
pd.DataFrame(baris)

Umumnya 8-bit hampir tanpa penurunan dan 4-bit masih sangat dekat, sedangkan 2-bit mulai merusak. Pada LLM besar pun berlaku hal serupa, sehingga model 7B bisa di-fine-tune dengan QLoRA di GPU 16 GB (notebook 02 menunjukkan konfigurasinya).

## Ringkasan

- **Prompting dulu**, **RAG untuk pengetahuan**, **fine-tuning untuk perilaku** (format, gaya, tugas sempit), dan selalu dibandingkan dengan baseline.
- Full fine-tuning butuh ±16 byte/parameter; **LoRA** membekukan bobot dan melatih adapter kecil, **QLoRA** menyimpan bobot beku dalam 4-bit.
- LoRA: `W + (α/r)·B·A`, `B` diinisialisasi nol (mulai identik dengan model asli), bisa di-**merge** tanpa tambahan latensi, dan bisa **dilepas** untuk kembali ke model asli.
- Adaptasi bisa menyebabkan **catastrophic forgetting**; adapter terpisah membuat model dasar tetap utuh.

## Latihan

1. **Rank vs data.** Ulangi eksperimen bagian 5 dengan `N_ADAPT` 200, 1.000, dan 5.000 untuk LoRA r=1, 4, 16, dan full fine-tuning. Buat grafik akurasi tugas B terhadap jumlah data. Kapan rank kecil sudah cukup?
2. **Target modules.** Pasang LoRA hanya pada `fc1`, lalu hanya pada `fc3`, lalu pada ketiganya (r=4). Layer mana yang paling penting untuk tugas "warna dibalik"? Mengapa masuk akal?
3. **Kapan LoRA gagal.** Ganti tugas B menjadi FashionMNIST dengan **warna dibalik** (`x = 1 - x`, latar putih). Bandingkan lagi LoRA r=1/4/16 dengan full fine-tuning. Kamu akan melihat LoRA jauh tertinggal. Jelaskan mengapa: untuk input `1 - x`, layer pertama harus berubah menjadi kira-kira `-W` (ditambah penyesuaian bias), sehingga `ΔW ≈ -2W`, yang rank-nya penuh. Apa artinya bagi asumsi low-rank LoRA?